# A Hierarchical Model of Honeybee Waggle-Dances on Horizontal and Vertical Combs
---

## Details

&ensp;AUTHOR:&emsp;   James Foster &emsp; 2026 10 14

MODIFIED:&emsp; James Foster &emsp; 2026 10 14

DESCRIPTION:&emsp; Fit a hierarchical maximum-likelihood von Mises to honeybee waggle-run angles relative to the feeder, comparing dances on horizontal and vertical combs at medium solar elevations.
Modified from [GUV_circGLMM_fit.ipynb](https://github.com/jjfosterlab/GUVdances), itself modified from [beetles.ipynb](https://github.com/jgraving/unwrap/notebooks/) from Jake Graving's `unwrap` package.

            
INPUTS:&emsp;    `0Data/hs_dance_reorg.csv`
            
OUTPUTS:&emsp;  `0Data/hs_dance_medel_VH.feather`, `2Results/HS_unwrap_*.nc`, `Plots/HS_VH_model_comparison.pdf`

REFERENCES:
- Sayin S, ... Graving JM, $\textit{et~al}$. (2025)
The behavioral mechanisms governing collective motion in swarming locusts.
$\textit{Science}$ 387,995-1000
DOI:[10.1126/science.adq7832](https://doi.org/10.1126/science.adq7832)

- Graving JM, Sayin S, Costelloe BR, Goldshtein A & Foster JJ. 
Unwrapping circular statistics with Bayesian multilevel models.

- Abril-Pla, O., Andreani, V., Carroll, C., Dong, L., Fonnesbeck, C. J., Kochurov, M., Kumar, R., Lao, J., Luhmann, C. C., Martin, O. A., et al. (2023).
`PyMC`: a modern, and comprehensive probabilistic programming framework in Python. 
*PeerJ Computer Science* 9, e1516.

- Capretto, T., Piho, C., Kumar, R., Westfall, J., Yarkoni, T., & Martin, O. (2022)
`Bambi`: A simple interface for fitting Bayesian linear models in Python.
*Journal of Statistical Software* 103(15), 1-29. doi:10.18637/jss.v103.i15

- Seyboldt, A., Carlson, E. L., & Carpenter, B. (2026).
Preconditioning Hamiltonian Monte Carlo by minimizing Fisher Divergence. 
*arXiv* preprint arXiv:2603.18845.

- Magnusson, M., Andersen, M.R., Jonasson, J., and Vehtari, A. (2020).
Leave-One-Out Cross-Validation for Bayesian Model Comparison in Large Data.
*arXiv* doi:10.48550/arXiv.2001.00980

---

# First set up the required packages
Don't forget to make sure the the kernel is running in a `conda` environment.

e.g. `conda create -n HS-env python=3.12 -c conda-forge -y`

To use `g++` (recommended for speed) make sure `m2w64-toolchain` is installed on Windows.

In [ ]:
import sys

# 20260520 - Modified for nutpie compliance with strict NumPy < 2 constraints
# 20261014 - Pinned unwrap (later commits require bambi>=0.15), nutpie and pyarrow (>=25 requires NumPy 2),
#            ipython & ipykernel (later versions show matplotlib figures as text only)
requirements = """
numpy>=1.26.4,<2.0.0
pandas==2.2.2
scipy==1.13.1
arviz==0.19.0
bambi==0.14.0
pymc==5.16.2
matplotlib==3.8.4
formulae==0.5.4
numba>=0.59.1,<=0.60.0
nutpie==0.13.2
netcdf4>=1.6.5,<1.7.0
h5py>=3.10.0,<4.0.0
git+https://github.com/jgraving/unwrap.git@3e701175b665a12f4a0dcda6643aef9ea4b7aecf
pyarrow==24.0.0
ipython==9.13.0
ipykernel==6.29.5
"""

# Write it to a file
with open("nutpie_unwrap_requirements.txt", "w") as f:
    f.write(requirements.strip() + "\n")

# Install everything from the file (on Windows "{sys.executable}" needs to be in quotes)
!"{sys.executable}" -m pip install -r nutpie_unwrap_requirements.txt --no-cache

%set_env PYTENSOR_FLAGS=blas__ldflags='-llapack -lblas -lcblas'

Report packages installed in enviroment.

In [ ]:
!"{sys.executable}" -m pip list

# Set up custom functions
Create a function for plotting circular model output in degrees clockwise.

In [ ]:
import matplotlib.pyplot as plt
import arviz as az

def plot_circular_trace(idata, 
                        vars, 
                        cvars, 
                        cvar_units='degrees', 
                        save_trans=False):
    """
    Transforms circular posterior variables from radians to degrees (inverted 
    for clockwise plotting) and plots the ArviZ trace.
    
    Parameters:
    -----------
    idata : arviz.InferenceData
        The ArviZ InferenceData object containing the posterior samples.
    vars : list of str
        The full list of variable names to plot in the trace.
    cvars : list of str
        The subset of variable names that are circular and need transformation.
    cvar_units : str, optional
        Units for the circular variables (e.g., 'degrees' or 'radians'). Default is 'degrees'.
    save_trans : bool, optional
        Whether to return the transformed InferenceData object. Default is True.
        
    Returns:
    --------
    idata_trans : arviz.InferenceData or None
        A copy of the InferenceData object with transformed circular variables if save_trans=True, 
        otherwise None.
    """
    # Transform the circular data for interpretability
    idata_trans = idata.copy()  # set up a variable for transformed data
    
    if cvar_units == 'degrees':
        for cvar in cvars:
            # Convert each to degrees and invert to plot clockwise
            idata_trans.posterior[cvar] = -180 * idata_trans.posterior[cvar] / np.pi 

    # Generate the trace plot
    az.plot_trace(
        idata_trans,
        var_names=vars,
        circ_var_names=cvars,
        circ_var_units=cvar_units
    ) 
    
    plt.tight_layout()
    plt.show()
    
    if save_trans:
        return idata_trans
    
    return None

# Prepare the data
## Import and format the data
Load the preprocessed data from CSV and select:
- **medium solar elevations**: $30° <$ `sun_el` $< 45°$ (as in `HS_ExploratoryPlots.Rmd`)
- dances in the **vertical** or **horizontal** conditions

Waggle-run angles relative to the feeder bearing (`feeder_angle`) are already in radians.
Relabel the two conditions to binary (`V`: 1 for vertical, 0 for horizontal).

In [ ]:
import os
import pandas as pd
import numpy as np
from unwrap.link.numpy import shifted_modulo

# Load the file
file_path = '0Data/hs_dance_reorg.csv'
df_all = pd.read_csv(file_path, sep=',')

# Select medium solar elevations and the vertical or horizontal conditions
medel_min, medel_max = 30, 45 # degrees, as in HS_ExploratoryPlots.Rmd
df = df_all[
    (df_all['sun_el'] > medel_min) & 
    (df_all['sun_el'] < medel_max) & 
    (df_all['condition'].isin(['vertical', 'horizontal']))
].copy()

# Create a binary indicator for comb orientation (1 for 'vertical', 0 for 'horizontal')
df['V'] = df['condition'].apply(lambda x: 1 if x == 'vertical' else 0)
# Rename waggle run to use capitalised label
df['Run'] = df['waggle_run']

# Convert the feeder angle column to numeric (already radians)
df['angle'] = pd.to_numeric(df['feeder_angle'], errors='coerce')  # Convert non-numeric values (e.g., 'NA') to NaN
# Sun azimuth relative to the feeder, for plotting (radians)
df['SunAz'] = np.deg2rad(df['sun_az'] - df['feeder_az'])

# Normalise all angles to be between -pi and pi (should be already)
df['angle'] = df['angle'].apply(lambda x: shifted_modulo(x) if pd.notnull(x) else x)
df['SunAz'] = df['SunAz'].apply(lambda x: shifted_modulo(x) if pd.notnull(x) else x)

# Keep only the relevant columns
df = df[['angle', 'ID', 'ID_d', 'V', 'Run', 'SunAz', 'sun_el']].copy()
df.reset_index(drop=True, inplace=True)

# Rename columns: change 'ID' to 'Individual' and 'ID_d' to 'Dance'
df = df.rename(columns={'ID': 'Individual', 'ID_d': 'Dance'})
df['Individual'] = df['Individual'].astype('category')
df['Dance'] = df['Dance'].astype('category')

# Keep the original labels for reference, then simplify to codes
labels = df[['Individual', 'Dance']].astype(str).rename(columns=lambda c: c + '_label')
df['Individual'] = df['Individual'].cat.codes + 1
df['Dance'] = df['Dance'].cat.codes + 1
df = pd.concat([df, labels], axis=1)

#save to align with raw data in R
df.to_feather('0Data/hs_dance_medel_VH.feather')

# Report sample sizes
print(df.groupby('V').agg(runs=('angle', 'size'),
                          dances=('Dance', 'nunique'),
                          individuals=('Individual', 'nunique')))
df.head()

## Inspect the data
Plot the data from the 6 individuals with the most waggle runs.
Each dance (`Dance`) is performed in a single condition, so dances are nested within individuals, but individuals may contribute dances in both conditions.

*N.B* `Pandas` indexes categories in alphabetical order, whereas `R` indexes them in order of appearance. Use the `Individual_label` and `Dance_label` columns of `hs_dance_medel_VH.feather` to align them.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import circmean, circstd
from matplotlib.lines import Line2D

# Helper function to compute the circular median.
def circ_median(angles):
    angles = np.sort(angles)
    diffs = np.abs(np.angle(np.exp(1j * (angles[:, None] - angles[None, :]))))
    sum_diffs = np.sum(diffs, axis=1)
    return angles[np.argmin(sum_diffs)]

# Helper function to scatter-plot data for one condition.
def plot_scatter(ax, data, angle_col, radial_col, color, label):
    if not data.empty:
        ax.scatter(data[angle_col], data[radial_col],
                   color=color, alpha=0.6, label=label)

# Helper function to compute and plot summary statistics.
# It plots a solid vertical line at the circular mean and a dashed vertical line at the circular median,
# both in the specified color. It also fills the region ± the circular standard deviation.
def plot_summary_stats(ax, angles, radial_max, color):
    # Compute circular mean, median, and standard deviation.
    mean_val = circmean(angles, high=np.pi, low=-np.pi)
    median_val = circ_median(angles)
    std_val = circstd(angles, high=np.pi, low=-np.pi, normalize=True)
    
    # Plot mean (solid) and median (dashed).
    ax.plot([mean_val, mean_val], [0, radial_max], color=color, linewidth=2)
    ax.plot([median_val, median_val], [0, radial_max], color=color, linewidth=2, linestyle='--')
    
    # Fill the region between mean minus std and mean plus std.
    theta_vals = np.linspace(mean_val - std_val, mean_val + std_val, 100)
    r_vals = np.linspace(0, radial_max, 100)
    Theta, R = np.meshgrid(theta_vals, r_vals)
    ax.contourf(Theta, R, np.ones_like(Theta), alpha=0.05, colors=[color]) # was alpha = 0.2, but covers too much data
    
    return mean_val, median_val, std_val

# Colours for each condition, as in HS_ExploratoryPlots.Rmd
col_h = 'darkgreen' # horizontal
col_v = 'darkred' # vertical

# Select the 6 individuals with the most waggle runs
plot_individuals = df['Individual'].value_counts().index[0:6]

# Create a grid of polar subplots.
n_plots = len(plot_individuals)
n_rows, n_cols = ( np.intc(n_plots/2) , 2  ) # Adjust grid dimensions as needed.
fig, axes = plt.subplots(n_rows, n_cols, subplot_kw={'projection': 'polar'}, figsize=(18, np.intc(n_plots*3)))
axes = axes.flatten()

radial_max = np.max(df['Run'])  # Maximum radial value (maximum runs)

# Plot for each individual.
for idx, individual in enumerate(plot_individuals):
    ax = axes[idx]
    # Set axes to geographic format (clockwise, 0° at the top = feeder direction)
    ax.set_theta_direction(-1)#reverse to clockwise
    ax.set_theta_zero_location('N')#place 0° at the top
    individual_data = df[df['Individual'] == individual]
    
    # Separate horizontal and vertical data (drop NaNs in angle).
    data_h = individual_data[individual_data['V'] == 0].dropna(subset=['angle'])
    data_v = individual_data[individual_data['V'] == 1].dropna(subset=['angle'])
    
    # Scatter-plot data.
    plot_scatter(ax, data_h, 'angle', 'Run', col_h, 'Horizontal')
    plot_scatter(ax, data_v, 'angle', 'Run', col_v, 'Vertical')
    
    #Plot the sun's azimuth relative to the feeder for each dance, for reference
    for sun_az in individual_data.groupby('Dance')['SunAz'].first():
        ax.plot([sun_az, sun_az], [0, radial_max], color='gold', linewidth=1)
    # Plot summary statistics for each condition.
    if not data_h.empty:
        plot_summary_stats(ax, data_h['angle'].values, radial_max, col_h)
    if not data_v.empty:
        plot_summary_stats(ax, data_v['angle'].values, radial_max, col_v)
    
    ax.set_title(f"Bee {individual_data['Individual_label'].iloc[0]} "
                 f"({individual_data['Dance'].nunique()} dances)", pad=30)
    ax.set_ylim(0, radial_max + 1)

# Create a shared custom legend.
legend_handles = [
    # For sun azimuth (gold)
    Line2D([0], [0], color = 'gold', lw = 1, label = 'Sun Azimuth (relative to feeder)'),
    # For horizontal
    Line2D([0], [0], marker='o', color='w', markerfacecolor=col_h, markersize=8, label='Horizontal Data'),
    Line2D([0], [0], color=col_h, lw=2, label='Horizontal Mean'),
    Line2D([0], [0], color=col_h, lw=2, linestyle='--', label='Horizontal Median'),
    # For vertical
    Line2D([0], [0], marker='o', color='w', markerfacecolor=col_v, markersize=8, label='Vertical Data'),
    Line2D([0], [0], color=col_v, lw=2, label='Vertical Mean'),
    Line2D([0], [0], color=col_v, lw=2, linestyle='--', label='Vertical Median'),
]

# Add one shared legend to the right of the figure.
fig.legend(handles=legend_handles, loc='center right', bbox_to_anchor=(1.10, 0.5), fontsize='medium')

plt.tight_layout()
plt.show()

# Fit a circular model with all potential effects
First fit the maximal model with effects of comb orientation (condition) on both mean direction ($\mu$) and directional consistency ($\kappa$).
Individuals may have danced in both conditions, so they receive random intercepts and random effects of condition.
Each dance was performed in a single condition, so dances (nested within individuals) receive random intercepts only.

$\mu \sim$ V + (1 + V |Individual) + (1 |Dance)

$\kappa \sim$ V + (1 + V |Individual) + (1 |Dance)

In [ ]:
import numpy as np
import pandas as pd
import bambi as bmb
import arviz as az
import matplotlib.pyplot as plt

from unwrap.link.bambi import LINKS
from unwrap.link.numpy import inverse_softplus
from unwrap.prior import CircularUniformPrior
from pymc import VonMises
from scipy.stats import vonmises, circstd

# Define the model formula.
formula_condition = bmb.Formula(
    "angle ~ V + (1 + V |Individual) + (1 |Dance)",
    "kappa ~ V + (1 + V |Individual) + (1 |Dance)",
)


# Specify priors for the model, following GUV_circGLMM_fit.ipynb.
priors_condition = {
    # Fixed effects for μ:
    "Intercept": CircularUniformPrior(), # Test
    "V": bmb.Prior("VonMises", mu=0, kappa=0.5), # Test
    # Individual-level effects for μ: informative prior, individuals close to pop mean circsd ~10°
    "1|Individual": bmb.Prior(
        "VonMises", mu=0, kappa=bmb.Prior("LogNormal", mu=np.log(30), sigma=0.60)
    ),
    "V|Individual": bmb.Prior(
        "VonMises", mu=0, kappa=bmb.Prior("LogNormal", mu=np.log(30), sigma=0.60)
    ),
    # Dance-level effects for μ: same informative prior, dances close to individual mean
    "1|Dance": bmb.Prior(
        "VonMises", mu=0, kappa=bmb.Prior("LogNormal", mu=np.log(30), sigma=0.60) # Test
    ),
    # Priors for the κ-model (inside the "kappa" dictionary):
    "kappa": {
        # Fixed effects for κ:
        "Intercept": bmb.Prior("Normal", mu=inverse_softplus(3), sigma=3.0),
        "V": bmb.Prior("Normal", mu=0, sigma=1.0), # Test
        # Individual-level effects for κ:
        "1|Individual": bmb.Prior(
            "Normal", mu=0, sigma=bmb.Prior("HalfStudentT", nu=3, sigma=1.0)
        ),
        "V|Individual": bmb.Prior(
            "Normal", mu=0, sigma=bmb.Prior("HalfStudentT", nu=3, sigma=1.0)
        ),
        # Dance-level effects for κ:
        "1|Dance": bmb.Prior(
            "Normal", mu=0, sigma=bmb.Prior("HalfStudentT", nu=3, sigma=1.0) # Test
        ),
    }
}

# Build the model using the formula and priors.
model_condition = bmb.Model(
    formula_condition,
    df,
    family="vonmises",
    priors=priors_condition,
    link={"mu": LINKS["shifted_modulo"], "kappa": LINKS["softplus"]},
    center_predictors=False,
    dropna=True,
)

print(model_condition)

# Fit the model using Bambi's native nutpie interface
idata_condition = model_condition.fit(
    draws=2000,
    tune=1000,
    nuts_sampler="nutpie",  #build and route via nutpie automatically
    target_accept=0.98
)

# Save model.
os.makedirs("2Results", exist_ok=True)
az.to_netcdf(idata_condition, "2Results/HS_unwrap_condition.nc")

## Check convergence heuristics

In [ ]:
#check convergence heuristics
az.summary(
    idata_condition, 
    var_names=["Intercept",
               "V", 
               "kappa_Intercept",
               "kappa_V",
                ],
    circ_var_names=["Intercept",
                    "V",
                    ]
)

In [ ]:
#check convergence heuristics on random effects
az.summary(
    idata_condition, 
    var_names=[
        "1|Individual_kappa",
        "V|Individual_kappa",
        "1|Dance_kappa",
        "kappa_1|Individual_sigma",
        "kappa_V|Individual_sigma",
        "kappa_1|Dance_sigma",
                ],
)

## Visualise output of the model

- The "Intercept" is the condition: **Horizontal**
- V is the effect of switching to: **Vertical**

In [ ]:
plot_circular_trace(
                    idata = idata_condition,
                    vars=[
                            "Intercept",
                            "V",
                            "kappa_Intercept",
                            "kappa_V",
                            "1|Individual_kappa",
                            "V|Individual_kappa",
                            "1|Dance_kappa",
                            "kappa_1|Individual_sigma",
                            "kappa_V|Individual_sigma",
                            "kappa_1|Dance_sigma",
                        ],
                    cvars = ["Intercept",
                            "V",
                            ]
)

In [ ]:
plot_circular_trace(
                    idata = idata_condition,
vars=[
        "1|Individual",
        "V|Individual",
        "kappa_1|Individual",
        "kappa_V|Individual",
       ],
cvars = [
        "1|Individual",
        "V|Individual",
        ]
)

# Alternative Hypotheses
We would like to test whether comb orientation affects dance direction and accuracy:
0. Are all data explained by individual and dance differences alone?
    - $\mu \sim$ 1
    - $\kappa \sim$ 1
1. Does comb orientation affect accuracy, but not mean direction?
    - $\mu \sim$ 1
    - $\kappa \sim$ condition
2. Does comb orientation affect both mean direction and accuracy?
    - $\mu \sim$ condition
    - $\kappa \sim$ condition
    - (already fitted)

In [ ]:
#model 0
formula_null = bmb.Formula(
    "angle ~ 1 + (1 |Individual) + (1 |Dance)",
    "kappa ~ 1 + (1 |Individual) + (1 |Dance)",
)

#model 1
formula_conditionKappa = bmb.Formula(
    "angle ~ 1 + (1 |Individual) + (1 |Dance)",
    "kappa ~ V + (1 + V |Individual) + (1 |Dance)",
)

#model 2 already fitted
    # "angle ~ V + (1 + V |Individual) + (1 |Dance)",
    # "kappa ~ V + (1 + V |Individual) + (1 |Dance)",

# =============================================================================
# 0. MODEL NULL PRIORS
# =============================================================================
priors_null = {
    "Intercept": bmb.Prior("VonMises", mu=0, kappa=1.0), # weak bias to the feeder direction (0°)
    "1|Individual": priors_condition["1|Individual"],
    "1|Dance": priors_condition["1|Dance"],
    "kappa": {
        "Intercept": priors_condition["kappa"]["Intercept"],
        "1|Individual": priors_condition["kappa"]["1|Individual"],
        "1|Dance": priors_condition["kappa"]["1|Dance"],
    }
}

# =============================================================================
# 1. MODEL CONDITION-KAPPA PRIORS
# =============================================================================
priors_conditionKappa = {
    "Intercept": priors_null["Intercept"],
    "1|Individual": priors_condition["1|Individual"],
    "1|Dance": priors_condition["1|Dance"],
    "kappa": priors_condition["kappa"],
}

#model 0
model_null = bmb.Model(
    formula_null,
    df,
    family="vonmises",
    priors=priors_null,
    link={"mu": LINKS["shifted_modulo"], "kappa": LINKS["softplus"]},
    center_predictors=False,
    dropna=True,
)

#model 1
model_conditionKappa = bmb.Model(
    formula_conditionKappa,
    df,
    family="vonmises",
    priors=priors_conditionKappa,
    link={"mu": LINKS["shifted_modulo"], "kappa": LINKS["softplus"]},
    center_predictors=False,
    dropna=True,
)

## Run Null model (0)
Null (Intercept, Individual-level and Dance-level random intercepts only)

In [ ]:
# Fit the model.
idata_null = model_null.fit(
    draws=2000,
    tune=1000,
    progressbar="split+stats",
    nuts_sampler="nutpie",  # build and route via nutpie automatically
    target_accept=0.98
)

# check convergence heuristics for fixed/population effects
az.summary(
    idata_null, 
    var_names=[
        "Intercept", 
        "kappa_Intercept",
        "1|Individual_kappa",
        "1|Dance_kappa",
        "kappa_1|Individual_sigma",
        "kappa_1|Dance_sigma",
    ],
    circ_var_names=[
        "Intercept",
    ]
)

## Run condition-kappa model (1)
$\mu$ no effects, $\kappa$ condition

In [ ]:
# Fit the model.
idata_conditionKappa = model_conditionKappa.fit(
    draws=2000,
    tune=1000,
    progressbar="split+stats",
    nuts_sampler="nutpie",  # build and route via nutpie automatically
    target_accept=0.98
)

az.summary(
    idata_conditionKappa, 
    var_names=[
        "Intercept", 
        "kappa_Intercept",
        "kappa_V",
        "1|Individual_kappa",
        "1|Dance_kappa",
        "kappa_1|Individual_sigma",
        "kappa_V|Individual_sigma",
        "kappa_1|Dance_sigma",
    ],
    circ_var_names=[
        "Intercept",
    ]
)

## Save all models

In [ ]:
# Ensure the results directory exists before saving
os.makedirs("2Results/unwrap_models", exist_ok=True)

# Dictionary mapping our fitted InferenceData objects to their descriptive filenames
models_to_save = {
    "model_0_null": idata_null,
    "model_1_conditionKappa": idata_conditionKappa,
    "model_2_condition": idata_condition  # Already fitted
}

# Systematically export each model to NetCDF format
for file_prefix, idata_obj in models_to_save.items():
    save_path = f"2Results/unwrap_models/HS_unwrap_{file_prefix}.nc"
    print(f"Exporting: {save_path}...")
    az.to_netcdf(idata_obj, save_path)

# Model comparison for hypothesis testing
## LOO-CV
Leave one out cross validation (LOO-CV) to calculate estimated log predictive density for each model (ELPD).

In [ ]:
import matplotlib as mpl

# Manually compute and attach the log-likelihood group to the existing idata
model_null.compute_log_likelihood(idata_null)
model_conditionKappa.compute_log_likelihood(idata_conditionKappa)
model_condition.compute_log_likelihood(idata_condition)

# Compute LOO
loo_null = az.loo(idata_null)
loo_conditionKappa = az.loo(idata_conditionKappa)
loo_condition = az.loo(idata_condition)

# Compare, ranking models from best to worst
comparison = az.compare(
    {
    'No effects': loo_null,
    'Condition kappa': loo_conditionKappa,
    'Condition': loo_condition,
    },
    ic="loo")

print('\n----- Model Comparison Ranking -----\n')
print(comparison.round(2))

# Ensure text is output as editable TrueType text elements in PDFs
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['font.sans-serif'] = ['Arial', 'Helvetica', 'Computer Modern Sans Serif']

# Visualize the comparison
ax_compare = az.plot_compare(comparison)
fig_compare = ax_compare.get_figure()
fig_compare.savefig("Plots/HS_VH_model_comparison.pdf", format="pdf", bbox_inches="tight")

# Extract & inspect posterior samples from maximal model

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import vonmises, circmean
from unwrap.link.numpy import shifted_modulo, softplus

# --------------------------------------------------
# Extract posterior samples for population-level parameters
# --------------------------------------------------
mu_intercept = idata_condition.posterior['Intercept'].values          
mu_slopeV    = idata_condition.posterior['V'].values               

kappa_intercept = idata_condition.posterior['kappa_Intercept'].values    
kappa_slopeV    = idata_condition.posterior['kappa_V'].values        

# --------------------------------------------------
# Prepare data grid
# --------------------------------------------------
theta = np.linspace(-np.pi, np.pi, 200)
theta_deg = theta * 180 / np.pi

# Compute population-level parameters for each condition
mu_pop_h_samples_all = shifted_modulo(mu_intercept.flatten())
mu_pop_v_samples_all = shifted_modulo((mu_intercept + mu_slopeV).flatten())

kappa_pop_h_samples_all = softplus(kappa_intercept.flatten())
kappa_pop_v_samples_all = softplus((kappa_intercept + kappa_slopeV).flatten())

# --------------------------------------------------
# Compute von Mises PDFs
# --------------------------------------------------
pdf_h = vonmises.pdf(theta[None, :], kappa=kappa_pop_h_samples_all[:, None], loc=mu_pop_h_samples_all[:, None])
pdf_v = vonmises.pdf(theta[None, :], kappa=kappa_pop_v_samples_all[:, None], loc=mu_pop_v_samples_all[:, None])

# --------------------------------------------------
# Setup 1x2 Subplot Grid
# --------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(15, 6), sharex=True, sharey=True)

conditions = [
    {"ax": axes[0], "pdf": pdf_h, "color": col_h, "title": "Horizontal (V=0)", "v_val": 0},
    {"ax": axes[1], "pdf": pdf_v, "color": col_v, "title": "Vertical (V=1)", "v_val": 1},
]

# Plot density and 95% Credible Intervals (2.5% to 97.5%) per panel
for cond in conditions:
    ax = cond["ax"]
    ax.plot(theta_deg, np.median(cond["pdf"], axis=0), color=cond["color"], lw=2.5, label="Posterior Median")
    ax.fill_between(theta_deg,
                    np.percentile(cond["pdf"], 2.5, axis=0),
                    np.percentile(cond["pdf"], 97.5, axis=0),
                    color=cond["color"], alpha=0.25, label="95% Credible Interval")
    ax.set_title(cond["title"], fontsize=12, fontweight="bold")
    ax.grid(True, alpha=0.3)

# --------------------------------------------------
# Calculate and Overlay Dance Summary Means
# --------------------------------------------------
for cond in conditions:
    dance_means = (df[df['V'] == cond["v_val"]]
                   .dropna(subset=['angle'])
                   .groupby('Dance')['angle']
                   .apply(lambda x: shifted_modulo(circmean(x, high=np.pi, low=-np.pi))))
    cond["ax"].scatter(
        dance_means * 180 / np.pi, np.zeros(len(dance_means)), 
        color=cond["color"], edgecolor='black', linewidth=0.5, 
        s=40, alpha=0.6, 
        label='Dance Means'
    )

# Global formatting details
for ax in axes.flat:
    ax.set_xlabel('Waggle-run angle relative to feeder (°)')
    ax.set_ylabel('Density')
    ax.label_outer() # Hides inner labels to avoid overlap clutter
    ax.legend(loc="upper right", frameon=True, facecolor="white", framealpha=0.8)

plt.suptitle('von Mises Probability Density, medium solar elevations', fontsize=16, fontweight="bold", y=0.98)
plt.tight_layout()
plt.show()